# Workplace Mental Health Disclosure And Support In The Technology Sector

## A Quantitative Survey Analysis

This capstone analyzes the OSMI Mental Health in Tech Survey from Kaggle. The project uses a public survey dataset to show a reproducible quantitative research workflow: data cleaning, segmentation, hypothesis testing, and logistic regression modeling.

## Executive Summary

The dataset contains **1,259 survey responses** about mental health and workplace support in the technology sector.

Main early findings:

- **637 respondents (50.6%)** reported seeking mental health treatment.
- **477 respondents (37.9%)** knew their employer provides mental health benefits.
- **782 respondents (62.1%)** reported at least some mental-health-related work interference.
- **516 respondents (41.0%)** were comfortable discussing mental health with a supervisor.
- Only **225 respondents (17.9%)** were comfortable discussing mental health with coworkers.
- The treatment-seeking logistic regression model achieved **ROC AUC = 0.891** on the test set.

The results suggest that treatment-seeking is common in this survey sample, but workplace disclosure comfort is much lower, especially with coworkers. Workplace benefits, care options, family history, and work interference are all statistically associated with treatment-seeking.

## Dataset Description

- **Source:** OSMI Mental Health in Tech Survey on Kaggle.
- **Raw file:** `data/osmi-mental-health-in-tech-survey/survey.csv`.
- **Rows:** 1,259.
- **Raw columns:** 27.
- **One row means:** one survey response.
- **Main topic:** mental health treatment, workplace support, stigma, disclosure comfort, and work interference in technology workplaces.

The dataset is observational and self-reported. This means the analysis can show associations, but it cannot prove cause and effect.

## Research Question And Hypotheses

**Research question:** What workplace and personal factors are associated with mental health treatment-seeking and disclosure comfort among technology workers?

Hypotheses:

1. Respondents who know their employer provides mental health benefits are more likely to seek treatment.
2. Respondents who expect negative workplace consequences are less comfortable discussing mental health with supervisors or coworkers.
3. Respondents whose mental health interferes with work are more likely to have sought treatment.
4. Workplace support indicators differ by company size, remote work status, and whether the employer is primarily a tech company.

## Techniques Used

This capstone uses three main techniques from the course:

- **Segmentation:** comparing treatment-seeking and disclosure comfort across workplace and personal groups.
- **Hypothesis testing:** using chi-square tests and Kruskal-Wallis tests to check whether group differences are statistically meaningful.
- **Regression/classification modeling:** using logistic regression to model whether a respondent sought mental health treatment.

A fourth supporting technique is dashboard-style visualization using chart outputs.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import chi2_contingency, kruskal
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TASK_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = TASK_DIR / "outputs"
CHART_DIR = OUTPUT_DIR / "charts"
REPORT_CHART_DIR = OUTPUT_DIR / "report_chart_assets"

clean_df = pd.read_csv(OUTPUT_DIR / "osmi_mental_health_cleaned.csv")
quality_df = pd.read_csv(OUTPUT_DIR / "osmi_data_quality_summary.csv")
executive_df = pd.read_csv(OUTPUT_DIR / "executive_summary_metrics.csv")
tests_df = pd.read_csv(OUTPUT_DIR / "hypothesis_test_results.csv")
model_perf_df = pd.read_csv(OUTPUT_DIR / "treatment_model_performance.csv")
model_coef_df = pd.read_csv(OUTPUT_DIR / "treatment_model_coefficients.csv")

treatment_by_benefits = pd.read_csv(OUTPUT_DIR / "treatment_by_benefits.csv")
treatment_by_care_options = pd.read_csv(OUTPUT_DIR / "treatment_by_care_options.csv")
treatment_by_family_history = pd.read_csv(OUTPUT_DIR / "treatment_by_family_history.csv")
treatment_by_work_interfere = pd.read_csv(OUTPUT_DIR / "treatment_by_work_interfere.csv")
treatment_by_company_size = pd.read_csv(OUTPUT_DIR / "treatment_by_company_size.csv")

clean_df.shape

(1259, 57)

## Data Quality And Cleaning

Survey data often contains messy values because respondents can enter information in different ways. The cleaning script made the analysis reproducible by:

- converting column names to `snake_case`;
- treating ages outside **18-75** as missing;
- grouping **49 raw gender values** into 4 broader analysis categories;
- creating binary variables such as `treatment_yes`, `benefits_yes`, and `family_history_yes`;
- creating ordered scores for work interference, leave difficulty, discussion comfort, and perceived consequences.

In [2]:
display(quality_df)

,metric,value
0,raw_rows,1259.0
1,raw_columns,27.0
2,cleaned_rows,1259.0
3,cleaned_columns,57.0
4,raw_missing_values,1892.0
5,invalid_or_out_of_range_age_values,8.0
6,raw_gender_unique_values,49.0
7,cleaned_gender_categories,4.0
8,treatment_yes_count,637.0
9,treatment_yes_percent,50.6


## Key Survey Metrics

These values provide the first overview of the sample before deeper analysis.

![Treatment-seeking overview](../outputs/report_chart_assets/figure_01_treatment_overview.png)

In [3]:
display(executive_df)

,metric,value
0,Total survey responses,1259
1,Sought mental health treatment,637 respondents (50.6%)
2,Know employer provides mental health benefits,477 respondents (37.9%)
3,Report any work interference,782 respondents (62.1%)
4,Comfortable discussing mental health with supe...,516 respondents (41.0%)
5,Comfortable discussing mental health with cowo...,225 respondents (17.9%)
6,Logistic model ROC AUC,0.891


## Segmentation: Treatment-Seeking By Workplace Support

Segmentation means dividing respondents into groups and comparing outcomes between those groups. Here, the outcome is whether a respondent reported seeking mental health treatment.

![Treatment-seeking by benefits and care options](../outputs/report_chart_assets/figure_02_benefits_and_care_options.png)

In [4]:
display(treatment_by_benefits)
display(treatment_by_care_options)

,benefits,respondents,treatment_count,treatment_rate,treatment_rate_percent
0,Yes,477,305,0.639413,63.9
1,No,374,181,0.483957,48.4
2,Don't know,408,151,0.370098,37.0


,care_options,respondents,treatment_count,treatment_rate,treatment_rate_percent
0,Yes,444,307,0.691441,69.1
1,No,501,207,0.413174,41.3
2,Not sure,314,123,0.391720,39.2


## Segmentation: Treatment-Seeking By Personal And Work Context

Family history and work interference are important context variables. They should not be interpreted as workplace policies, but they help explain why treatment-seeking may differ between respondents.

![Treatment-seeking by work interference](../outputs/report_chart_assets/figure_03_treatment_by_work_interference.png)

In [5]:
display(treatment_by_family_history)
display(treatment_by_work_interfere)

,family_history,respondents,treatment_count,treatment_rate,treatment_rate_percent
0,Yes,492,365,0.741870,74.2
1,No,767,272,0.354628,35.5


,work_interfere,respondents,treatment_count,treatment_rate,treatment_rate_percent
0,Often,144,123,0.854167,85.4
1,Sometimes,465,358,0.769892,77.0
2,Rarely,173,122,0.705202,70.5
3,Never,213,30,0.140845,14.1
4,Missing,264,4,0.015152,1.5


## Hypothesis Testing

A hypothesis test checks whether an observed difference is large enough that it is unlikely to be only random noise.

For categorical variables, this notebook uses the **chi-square test**. Example: it tests whether treatment-seeking differs across benefit-awareness groups.

For ordered scores across multiple company-size groups, this notebook uses the **Kruskal-Wallis test**. This is useful when we compare ordinal or non-normal values across more than two groups.

In [6]:
def chi_square_test(df, row_col, target_col):
    table = pd.crosstab(df[row_col], df[target_col])
    chi2, p_value, dof, expected = chi2_contingency(table)
    return {
        "test": "chi-square",
        "question": f"Association between {row_col} and {target_col}",
        "variable_1": row_col,
        "variable_2": target_col,
        "chi2": round(float(chi2), 4),
        "degrees_of_freedom": int(dof),
        "p_value": round(float(p_value), 6),
        "significant_at_0_05": bool(p_value < 0.05),
        "table_rows": int(table.shape[0]),
        "table_columns": int(table.shape[1]),
    }

def kruskal_test(df, group_col, score_col):
    usable = df[[group_col, score_col]].dropna()
    groups = [group[score_col].values for _, group in usable.groupby(group_col)]
    statistic, p_value = kruskal(*groups)
    return {
        "test": "kruskal-wallis",
        "question": f"Difference in {score_col} across {group_col}",
        "variable_1": group_col,
        "variable_2": score_col,
        "statistic": round(float(statistic), 4),
        "p_value": round(float(p_value), 6),
        "significant_at_0_05": bool(p_value < 0.05),
        "groups": int(usable[group_col].nunique()),
        "usable_rows": int(len(usable)),
    }

calculated_tests_df = pd.DataFrame(
    [
        chi_square_test(clean_df, "benefits", "treatment"),
        chi_square_test(clean_df, "care_options", "treatment"),
        chi_square_test(clean_df, "family_history", "treatment"),
        chi_square_test(clean_df, "work_interfere", "treatment"),
        chi_square_test(clean_df, "mental_health_consequence", "supervisor"),
        chi_square_test(clean_df, "mental_health_consequence", "coworkers"),
        chi_square_test(clean_df, "remote_work", "benefits"),
        chi_square_test(clean_df, "tech_company", "benefits"),
        kruskal_test(clean_df, "company_size_label", "discussion_comfort_score"),
        kruskal_test(clean_df, "company_size_label", "leave_difficulty_score"),
    ]
)

display(calculated_tests_df)

,test,question,variable_1,variable_2,chi2,degrees_of_freedom,p_value,significant_at_0_05,table_rows,table_columns,statistic,groups,usable_rows
0,chi-square,Association between benefits and treatment,benefits,treatment,64.8386,2.0,0.000000,True,3.0,2.0,NaN,NaN,NaN
1,chi-square,Association between care_options and treatment,care_options,treatment,94.7587,2.0,0.000000,True,3.0,2.0,NaN,NaN,NaN
2,chi-square,Association between family_history and treatment,family_history,treatment,178.2668,1.0,0.000000,True,2.0,2.0,NaN,NaN,NaN
3,chi-square,Association between work_interfere and treatment,work_interfere,treatment,594.9243,4.0,0.000000,True,5.0,2.0,NaN,NaN,NaN
4,chi-square,Association between mental_health_consequence ...,mental_health_consequence,supervisor,461.6603,4.0,0.000000,True,3.0,3.0,NaN,NaN,NaN
5,chi-square,Association between mental_health_consequence ...,mental_health_consequence,coworkers,285.5339,4.0,0.000000,True,3.0,3.0,NaN,NaN,NaN
6,chi-square,Association between remote_work and benefits,remote_work,benefits,12.4816,2.0,0.001948,True,2.0,3.0,NaN,NaN,NaN
7,chi-square,Association between tech_company and benefits,tech_company,benefits,9.4468,2.0,0.008885,True,2.0,3.0,NaN,NaN,NaN
8,kruskal-wallis,Difference in discussion_comfort_score across ...,company_size_label,discussion_comfort_score,NaN,NaN,0.000238,True,NaN,NaN,23.7876,6.0,1259.0
9,kruskal-wallis,Difference in leave_difficulty_score across co...,company_size_label,leave_difficulty_score,NaN,NaN,0.065674,False,NaN,NaN,10.3592,6.0,1259.0


The table above is calculated directly in this notebook. The saved table below is the same output stored in the project files and used by the report, dashboard, and presentation.

In [7]:
display(tests_df)

,test,question,variable_1,variable_2,chi2,degrees_of_freedom,p_value,significant_at_0_05,table_rows,table_columns,statistic,groups,usable_rows
0,chi-square,Association between benefits and treatment,benefits,treatment,64.8386,2.0,0.000000,True,3.0,2.0,NaN,NaN,NaN
1,chi-square,Association between care_options and treatment,care_options,treatment,94.7587,2.0,0.000000,True,3.0,2.0,NaN,NaN,NaN
2,chi-square,Association between family_history and treatment,family_history,treatment,178.2668,1.0,0.000000,True,2.0,2.0,NaN,NaN,NaN
3,chi-square,Association between work_interfere and treatment,work_interfere,treatment,594.9243,4.0,0.000000,True,5.0,2.0,NaN,NaN,NaN
4,chi-square,Association between mental_health_consequence ...,mental_health_consequence,supervisor,461.6603,4.0,0.000000,True,3.0,3.0,NaN,NaN,NaN
5,chi-square,Association between mental_health_consequence ...,mental_health_consequence,coworkers,285.5339,4.0,0.000000,True,3.0,3.0,NaN,NaN,NaN
6,chi-square,Association between remote_work and benefits,remote_work,benefits,12.4816,2.0,0.001948,True,2.0,3.0,NaN,NaN,NaN
7,chi-square,Association between tech_company and benefits,tech_company,benefits,9.4468,2.0,0.008885,True,2.0,3.0,NaN,NaN,NaN
8,kruskal-wallis,Difference in discussion_comfort_score across ...,company_size_label,discussion_comfort_score,NaN,NaN,0.000238,True,NaN,NaN,23.7876,6.0,1259.0
9,kruskal-wallis,Difference in leave_difficulty_score across co...,company_size_label,leave_difficulty_score,NaN,NaN,0.065674,False,NaN,NaN,10.3592,6.0,1259.0


## Hypothesis Test Interpretation

The main tests found statistically significant associations at the 0.05 level:

- Employer mental health benefits and treatment-seeking: **p < 0.001**.
- Care options and treatment-seeking: **p < 0.001**.
- Family history and treatment-seeking: **p < 0.001**.
- Work interference and treatment-seeking: **p < 0.001**.
- Expected mental health consequences and supervisor discussion comfort: **p < 0.001**.
- Expected mental health consequences and coworker discussion comfort: **p < 0.001**.

Company size was significantly associated with discussion comfort (**p = 0.000238**), but not with leave difficulty at the 0.05 level (**p = 0.065674**).

![Supervisor discussion comfort by expected consequences](../outputs/report_chart_assets/figure_04_supervisor_comfort_by_consequence.png)

![Treatment-seeking by company size](../outputs/report_chart_assets/figure_05_treatment_by_company_size.png)

## Logistic Regression Model

Logistic regression is used when the target variable has two outcomes. In this project, the target is:

- `1`: respondent sought mental health treatment;
- `0`: respondent did not seek mental health treatment.

The model uses workplace support, work context, family history, age, and gender variables to estimate which factors are associated with treatment-seeking.

In [8]:
model_df = clean_df[
    [
        "treatment_yes",
        "age_clean",
        "gender_clean",
        "family_history",
        "work_interfere",
        "no_employees",
        "remote_work",
        "tech_company",
        "benefits",
        "care_options",
        "wellness_program",
        "seek_help",
        "anonymity",
        "leave",
        "mental_health_consequence",
        "coworkers",
        "supervisor",
        "obs_consequence",
    ]
].copy()

x = model_df.drop(columns=["treatment_yes"])
y = model_df["treatment_yes"]

numeric_features = ["age_clean"]
categorical_features = [col for col in x.columns if col not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first")),
                ]
            ),
            categorical_features,
        ),
    ]
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)
logistic_model.fit(x_train, y_train)

y_pred = logistic_model.predict(x_test)
y_prob = logistic_model.predict_proba(x_test)[:, 1]
report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)

calculated_model_perf_df = pd.DataFrame(
    [
        {
            "model": "Logistic regression",
            "target": "treatment_yes",
            "train_rows": len(x_train),
            "test_rows": len(x_test),
            "accuracy": round(float(accuracy_score(y_test, y_pred)), 3),
            "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 3),
            "precision_treatment_yes": round(report["1"]["precision"], 3),
            "recall_treatment_yes": round(report["1"]["recall"], 3),
            "f1_treatment_yes": round(report["1"]["f1-score"], 3),
        }
    ]
)

feature_names = logistic_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = logistic_model.named_steps["classifier"].coef_[0]
calculated_model_coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coefficients,
        "odds_ratio": np.exp(coefficients),
    }
)
calculated_model_coef_df["abs_coefficient"] = calculated_model_coef_df["coefficient"].abs()
calculated_model_coef_df = calculated_model_coef_df.sort_values(
    "abs_coefficient", ascending=False
).drop(columns="abs_coefficient")
calculated_model_coef_df["coefficient"] = calculated_model_coef_df["coefficient"].round(4)
calculated_model_coef_df["odds_ratio"] = calculated_model_coef_df["odds_ratio"].round(3)

display(calculated_model_perf_df)
display(calculated_model_coef_df.head(15))

,model,target,train_rows,test_rows,accuracy,roc_auc,precision_treatment_yes,recall_treatment_yes,f1_treatment_yes
0,Logistic regression,treatment_yes,944,315,0.819,0.891,0.797,0.862,0.828


,feature,coefficient,odds_ratio
6,categorical__work_interfere_Often,3.9898,54.042
8,categorical__work_interfere_Sometimes,3.6472,38.368
7,categorical__work_interfere_Rarely,3.0599,21.326
4,categorical__family_history_Yes,1.0497,2.857
33,categorical__coworkers_Yes,0.9821,2.670
5,categorical__work_interfere_Never,0.9249,2.522
1,categorical__gender_clean_Male,-0.6928,0.500
19,categorical__care_options_Yes,0.6083,1.837
22,categorical__seek_help_No,-0.5213,0.594
26,categorical__leave_Somewhat difficult,0.5028,1.653


The model outputs above are calculated directly in this notebook. The saved tables below are the matching project outputs used by the report, dashboard, and presentation.

In [9]:
display(model_perf_df)
display(model_coef_df.head(15))

,model,target,train_rows,test_rows,accuracy,roc_auc,precision_treatment_yes,recall_treatment_yes,f1_treatment_yes
0,Logistic regression,treatment_yes,944,315,0.819,0.891,0.797,0.862,0.828


,feature,coefficient,odds_ratio
0,categorical__work_interfere_Often,3.9898,54.042
1,categorical__work_interfere_Sometimes,3.6472,38.368
2,categorical__work_interfere_Rarely,3.0599,21.326
3,categorical__family_history_Yes,1.0497,2.857
4,categorical__coworkers_Yes,0.9821,2.670
5,categorical__work_interfere_Never,0.9249,2.522
6,categorical__gender_clean_Male,-0.6928,0.500
7,categorical__care_options_Yes,0.6083,1.837
8,categorical__seek_help_No,-0.5213,0.594
9,categorical__leave_Somewhat difficult,0.5028,1.653


## Model Interpretation

The model achieved **ROC AUC = 0.891**, which means it separated treatment-seeking and non-treatment-seeking respondents well on the test data.

![Logistic regression model performance](../outputs/report_chart_assets/figure_06_model_performance.png)

The strongest positive model signals included:

- frequent work interference;
- family history of mental illness;
- comfort discussing mental health with coworkers;
- knowing that employer care options are available;
- knowing that employer mental health benefits are available.

Important caution: this is still association, not causation. For example, the model does not prove that work interference causes treatment. It shows that work interference is strongly related to treatment-seeking in this survey sample.

## Dashboard-Ready Chart Outputs

The main dashboard-style visuals are now available inside this notebook and in the written report as static PNG chart snippets. The project also keeps supplementary interactive HTML charts in `outputs/charts/` and a standalone HTML dashboard in `outputs/dashboard/`.

Current chart files:

- `treatment_overview.html`
- `treatment_by_benefits.html`
- `treatment_by_work_interference.html`
- `supervisor_comfort_by_consequence.html`

These charts support the dashboard, presentation, and written report. For formal submission, the notebook and report should be treated as the primary visual artifacts; the standalone HTML dashboard is supplementary.

In [10]:
chart_files = sorted(path.name for path in CHART_DIR.glob("*.html"))
chart_files

['supervisor_comfort_by_consequence.html',
 'treatment_by_benefits.html',
 'treatment_by_work_interference.html',
 'treatment_overview.html']

## Recommendations

Based on the current analysis, technology employers should:

1. Make mental health benefits and care options easier to understand, because benefit awareness is associated with treatment-seeking.
2. Strengthen anonymity and confidentiality communication, because disclosure risk appears central to supervisor and coworker comfort.
3. Train managers to discuss mental health safely, because only 41.0% of respondents were comfortable discussing mental health with supervisors.
4. Reduce stigma in team culture, because only 17.9% of respondents were comfortable discussing mental health with coworkers.
5. Monitor work interference as an early warning signal, because work interference is strongly associated with treatment-seeking.

## Limitations

- The dataset is observational, so the analysis cannot prove cause and effect.
- Respondents are not a perfectly random sample of all technology workers.
- The data is self-reported, so it may include recall bias or social desirability bias.
- Some variables have missing values, especially `work_interfere`.
- Gender was grouped from many raw self-described values, which simplifies identity categories for analysis.
- The survey is from one dataset and should not be generalized without caution.

## Final Conclusion

The analysis shows that mental health treatment-seeking is common in this technology-sector survey sample, but workplace disclosure comfort is much lower. Workplace benefits, care options, family history, work interference, and perceived consequences are all meaningfully associated with treatment-seeking or disclosure comfort.

For the capstone story, the most important insight is the gap between private action and workplace openness: **50.6%** of respondents sought treatment, but only **41.0%** were comfortable discussing mental health with a supervisor and only **17.9%** were comfortable discussing it with coworkers.